# Deep Bidirectional LSTM for Hindi Syllable Grouping & Scansion

This notebook implements the **next-generation sequence model** proposed in `syllable_grouping_training.ipynb`:
A **Deep Bidirectional LSTM (BiLSTM)** sequence labeler with multi-input feature embeddings.

### Evolution of Models for this Task:
1. **Rule-Based Chandas Algorithm**: Rigid right-to-left pairing $\rightarrow$ **57.3%** exact sequence match.
2. **Baseline Logistic Regression**: N-gram context window $\rightarrow$ **90.4%** exact sequence match.
3. **Deep BiLSTM Sequence Model (This Notebook)**: Dense neural embeddings + Bidirectional recurrent context $\rightarrow$ **~96.9%** exact sequence match.

---

### Model Architecture:
- **Syllable Akshara Embedding**: Dense representation of individual Hindi syllable graphemes.
- **Weight Embedding**: Dense representation of syllable prosodic weight (Laghu = 1, Guru = 2).
- **Heaviness Embedding**: Distinguishes intrinsic heaviness (long vowel/nasal) from positional heaviness (closed by conjunct cluster).
- **Relative Position Feature**: Normalized continuous position within the word.
- **2-Layer Bidirectional LSTM**: Processes forward and backward context simultaneously across the word.
- **Linear Projection Head**: Predicts BIO boundary probability distributions for each syllable.


In [1]:
# Install required dependencies
# Run this cell if executing on Google Colab or in a fresh environment
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn


In [2]:
import os
import sys
import time
import math
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Set seed for complete reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using compute device: {device}")


Using compute device: cpu


## 1. Data Loading & Sequence Construction

We load `data.csv` (generated from `corpus.json`), which contains all 40,054 syllables from the 13,196 valid corpus words.
The data is grouped into distinct word sequences and split into **80% Train**, **10% Validation**, and **10% Test** sets at the sequence level.


In [3]:
# Ensure data.csv exists, or generate it automatically from corpus.json
DATA_PATH = "data.csv"

if not os.path.exists(DATA_PATH):
    print(f"'{DATA_PATH}' not found. Attempting to build from corpus.json...")
    from prepare_data_csv import build_dataframe
    df = build_dataframe("corpus.json")
    df.to_csv(DATA_PATH, index=False, encoding="utf-8")
else:
    df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")
display(df.head(6))

# Group syllables into distinct word sequences
sequences = []
for seq_id, g in df.groupby("sequence_id", sort=False):
    sequences.append({
        "id": seq_id,
        "syllables": g["syllable"].tolist(),
        "weights": g["weight"].tolist(),
        "heaviness": g["heaviness"].tolist(),
        "labels": g["label"].tolist()
    })

n_total = len(sequences)
indices = np.arange(n_total)
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

n_train = int(0.80 * n_total)
n_val = int(0.10 * n_total)

train_seqs = [sequences[i] for i in indices[:n_train]]
val_seqs = [sequences[i] for i in indices[n_train:n_train + n_val]]
test_seqs = [sequences[i] for i in indices[n_train + n_val:]]

print(f"Total word sequences : {n_total}")
print(f"  Train sequences    : {len(train_seqs)} ({sum(len(s['syllables']) for s in train_seqs):,} syllables)")
print(f"  Val sequences      : {len(val_seqs)} ({sum(len(s['syllables']) for s in val_seqs):,} syllables)")
print(f"  Test sequences     : {len(test_seqs)} ({sum(len(s['syllables']) for s in test_seqs):,} syllables)")


Dataset shape: (40054, 6)


,sequence_id,position,syllable,weight,heaviness,label
0,कंकड़,0,कं,2,intrinsic,B
1,कंकड़,1,क,1,light,B
2,कंकड़,2,ड़,1,light,I
3,कंकण,0,कं,2,intrinsic,B
4,कंकण,1,क,1,light,B
5,कंकण,2,ण,1,light,I


Total word sequences : 13196
  Train sequences    : 10556 (32,027 syllables)
  Val sequences      : 1319 (4,052 syllables)
  Test sequences     : 1321 (3,975 syllables)


In [4]:
# Build Syllable Vocabulary from Training Data
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

syl_vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for seq in train_seqs:
    for s in seq["syllables"]:
        if s not in syl_vocab:
            syl_vocab[s] = len(syl_vocab)

print(f"Syllable Vocabulary size: {len(syl_vocab)}")

# Feature mapping dictionaries
weight_map = {1: 0, 2: 1, "<PAD>": 2}
heavy_map = {"light": 0, "intrinsic": 1, "positional": 2, "<PAD>": 3}
label_map = {"B": 0, "I": 1}
rev_label_map = {0: "B", 1: "I"}


class ScansionSequenceDataset(Dataset):
    def __init__(self, seqs):
        self.seqs = seqs

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        s = self.seqs[idx]
        L = len(s["syllables"])
        syl_ids = [syl_vocab.get(x, syl_vocab[UNK_TOKEN]) for x in s["syllables"]]
        w_ids = [weight_map.get(x, 0) for x in s["weights"]]
        h_ids = [heavy_map.get(x, 0) for x in s["heaviness"]]
        pos_norm = [i / max(1, L - 1) for i in range(L)]
        lbl_ids = [label_map[x] for x in s["labels"]]

        return {
            "syl": torch.tensor(syl_ids, dtype=torch.long),
            "weight": torch.tensor(w_ids, dtype=torch.long),
            "heavy": torch.tensor(h_ids, dtype=torch.long),
            "pos": torch.tensor(pos_norm, dtype=torch.float).unsqueeze(-1),
            "label": torch.tensor(lbl_ids, dtype=torch.long),
            "length": L,
            "raw": s
        }


def pad_collate_fn(batch):
    lengths = [b["length"] for b in batch]
    max_len = max(lengths)
    B = len(batch)

    syl = torch.zeros(B, max_len, dtype=torch.long)
    weight = torch.full((B, max_len), weight_map["<PAD>"], dtype=torch.long)
    heavy = torch.full((B, max_len), heavy_map["<PAD>"], dtype=torch.long)
    pos = torch.zeros(B, max_len, 1, dtype=torch.float)
    label = torch.full((B, max_len), -100, dtype=torch.long)  # -100 ignored by CrossEntropyLoss

    for i, b in enumerate(batch):
        L = b["length"]
        syl[i, :L] = b["syl"]
        weight[i, :L] = b["weight"]
        heavy[i, :L] = b["heavy"]
        pos[i, :L] = b["pos"]
        label[i, :L] = b["label"]

    return syl, weight, heavy, pos, label, lengths, [b["raw"] for b in batch]


BATCH_SIZE = 64
train_loader = DataLoader(ScansionSequenceDataset(train_seqs), batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_collate_fn)
val_loader = DataLoader(ScansionSequenceDataset(val_seqs), batch_size=128, shuffle=False, collate_fn=pad_collate_fn)
test_loader = DataLoader(ScansionSequenceDataset(test_seqs), batch_size=128, shuffle=False, collate_fn=pad_collate_fn)


Syllable Vocabulary size: 1477


## 2. Deep Bidirectional LSTM Architecture

The model combines:
1. **Multi-Input Feature Embeddings**:
   - Syllable Embedding ($d=64$)
   - Weight Embedding ($d=16$)
   - Heaviness Type Embedding ($d=16$)
   - Continuous Relative Position ($d=1$)
   Total input dimension = $64 + 16 + 16 + 1 = 97$
2. **Bidirectional Recurrent Layer**:
   - 2 stacked BiLSTM layers ($h=64$ in each direction $\rightarrow 128$-dimensional hidden representation)
   - Recurrent Dropout = $0.20$
3. **Sequence Classification Head**:
   - Linear projection $128 \rightarrow 2$ output logits (`B` vs. `I`)


In [5]:
class BiLSTMScansionModel(nn.Module):
    def __init__(self, vocab_size, syl_dim=64, w_dim=16, h_dim=16, hidden_dim=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.syl_emb = nn.Embedding(vocab_size, syl_dim, padding_idx=0)
        self.w_emb = nn.Embedding(3, w_dim, padding_idx=2)
        self.h_emb = nn.Embedding(4, h_dim, padding_idx=3)
        self.drop = nn.Dropout(dropout)

        in_dim = syl_dim + w_dim + h_dim + 1
        self.lstm = nn.LSTM(
            input_size=in_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.classifier = nn.Linear(hidden_dim * 2, 2)

    def forward(self, syl, weight, heavy, pos):
        # Embed all feature representations
        e_syl = self.syl_emb(syl)
        e_w = self.w_emb(weight)
        e_h = self.h_emb(heavy)

        # Concatenate dense embeddings with relative position feature
        features = torch.cat([e_syl, e_w, e_h, pos], dim=-1)
        features = self.drop(features)

        # Pass through Bidirectional LSTM
        lstm_out, _ = self.lstm(features)
        lstm_out = self.drop(lstm_out)

        # Linear classification
        logits = self.classifier(lstm_out)
        return logits


model = BiLSTMScansionModel(len(syl_vocab)).to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal Trainable Parameters: {param_count:,}")


BiLSTMScansionModel(
  (syl_emb): Embedding(1477, 64, padding_idx=0)
  (w_emb): Embedding(3, 16, padding_idx=2)
  (h_emb): Embedding(4, 16, padding_idx=3)
  (drop): Dropout(p=0.2, inplace=False)
  (lstm): LSTM(97, 64, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)

Total Trainable Parameters: 277,682


## 3. Training & Validation Loop

We optimize using **AdamW** with weight decay and track both **Token-Level Accuracy** and **Exact Word Sequence Accuracy** on the validation set after every epoch. The checkpoint with the highest validation exact sequence match is automatically saved.


In [6]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.003, weight_decay=1e-4)

EPOCHS = 6
best_val_seq_acc = 0.0
os.makedirs("trained_model", exist_ok=True)
CHECKPOINT_PATH = "trained_model/bilstm_grouping_model.pt"

history = {
    "epoch": [], "train_loss": [], "val_token_acc": [], "val_seq_acc": []
}

print(f"{'Epoch':<7} | {'Train Loss':<12} | {'Val Token Acc':<15} | {'Val Exact Seq Acc':<18} | {'Time':<8}")
print("-" * 72)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running_loss = 0.0

    for syl, w, h, p, y, lengths, _ in train_loader:
        syl = syl.to(device)
        w = w.to(device)
        h = h.to(device)
        p = p.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(syl, w, h, p)
        loss = criterion(logits.view(-1, 2), y.view(-1))
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # Validation evaluation
    model.eval()
    correct_tokens = 0
    total_tokens = 0
    exact_seq_matches = 0
    total_seqs = 0

    with torch.no_grad():
        for syl, w, h, p, y, lengths, _ in val_loader:
            syl = syl.to(device)
            w = w.to(device)
            h = h.to(device)
            p = p.to(device)
            y = y.to(device)

            logits = model(syl, w, h, p)
            preds = logits.argmax(dim=-1)

            for i, L in enumerate(lengths):
                pred_seq = preds[i, :L]
                gold_seq = y[i, :L]

                correct_tokens += (pred_seq == gold_seq).sum().item()
                total_tokens += L

                if torch.equal(pred_seq, gold_seq):
                    exact_seq_matches += 1
                total_seqs += 1

    val_token_acc = correct_tokens / total_tokens
    val_seq_acc = exact_seq_matches / total_seqs
    elapsed = time.time() - t0

    history["epoch"].append(epoch)
    history["train_loss"].append(avg_train_loss)
    history["val_token_acc"].append(val_token_acc)
    history["val_seq_acc"].append(val_seq_acc)

    print(f"{epoch:<7} | {avg_train_loss:<12.4f} | {val_token_acc:<15.4f} | {val_seq_acc:<18.4f} | {elapsed:<6.1f}s")

    # Save best checkpoint
    if val_seq_acc > best_val_seq_acc:
        best_val_seq_acc = val_seq_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "syl_vocab": syl_vocab,
            "weight_map": weight_map,
            "heavy_map": heavy_map,
            "label_map": label_map,
            "rev_label_map": rev_label_map,
            "config": {
                "vocab_size": len(syl_vocab),
                "syl_dim": 64,
                "w_dim": 16,
                "h_dim": 16,
                "hidden_dim": 64,
                "num_layers": 2
            }
        }, CHECKPOINT_PATH)

print(f"\nBest validation exact sequence accuracy: {best_val_seq_acc * 100:.2f}%")
print(f"Saved best model checkpoint to {CHECKPOINT_PATH}")


Epoch   | Train Loss   | Val Token Acc   | Val Exact Seq Acc  | Time    
------------------------------------------------------------------------
1       | 0.1332       | 0.9724          | 0.9280             | 3.1   s
2       | 0.0561       | 0.9820          | 0.9500             | 3.0   s
3       | 0.0420       | 0.9842          | 0.9553             | 2.6   s
4       | 0.0321       | 0.9867          | 0.9636             | 2.9   s
5       | 0.0294       | 0.9847          | 0.9560             | 2.6   s
6       | 0.0251       | 0.9874          | 0.9644             | 2.5   s

Best validation exact sequence accuracy: 96.44%
Saved best model checkpoint to trained_model/bilstm_grouping_model.pt


## 4. Evaluation on Unseen Test Set

We load the saved best checkpoint and evaluate on the 1,321 held-out test words (4,016 syllables) that were never seen during training or validation.


In [7]:
# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

all_gold = []
all_pred = []
test_seq_results = []

with torch.no_grad():
    for syl, w, h, p, y, lengths, raw_batch in test_loader:
        syl = syl.to(device)
        w = w.to(device)
        h = h.to(device)
        p = p.to(device)

        logits = model(syl, w, h, p)
        preds = logits.argmax(dim=-1)

        for i, L in enumerate(lengths):
            p_seq = [rev_label_map[idx] for idx in preds[i, :L].cpu().numpy()]
            g_seq = raw_batch[i]["labels"]

            all_gold.extend(g_seq)
            all_pred.extend(p_seq)

            # Reconstruct groups from BIO tags
            def reconstruct_groups(tokens, tags):
                groups, curr = [], []
                for tok, tag in zip(tokens, tags):
                    if tag == "B" and curr:
                        groups.append("".join(curr))
                        curr = []
                    curr.append(tok)
                if curr:
                    groups.append("".join(curr))
                return groups

            gold_groups = reconstruct_groups(raw_batch[i]["syllables"], g_seq)
            pred_groups = reconstruct_groups(raw_batch[i]["syllables"], p_seq)

            test_seq_results.append({
                "word": raw_batch[i]["id"],
                "syllables": raw_batch[i]["syllables"],
                "gold_labels": g_seq,
                "pred_labels": p_seq,
                "gold_groups": gold_groups,
                "pred_groups": pred_groups,
                "exact_match": (p_seq == g_seq)
            })

exact_match_rate = np.mean([r["exact_match"] for r in test_seq_results])
token_acc = accuracy_score(all_gold, all_pred)

print("=" * 65)
print("TEST SET FINAL RESULTS")
print("=" * 65)
print(f"Token-Level Classification Accuracy : {token_acc * 100:.2f}%")
print(f"Exact Word Sequence Grouping Match : {exact_match_rate * 100:.2f}%")
print("=" * 65)

print("\nClassification Report:")
print(classification_report(all_gold, all_pred, digits=4))

print("Confusion Matrix:")
cm = confusion_matrix(all_gold, all_pred, labels=["B", "I"])
display(pd.DataFrame(cm, index=["Actual B", "Actual I"], columns=["Pred B", "Pred I"]))


TEST SET FINAL RESULTS
Token-Level Classification Accuracy : 99.14%
Exact Word Sequence Grouping Match : 97.58%

Classification Report:
              precision    recall  f1-score   support

           B     0.9956    0.9929    0.9942      2959
           I     0.9795    0.9872    0.9833      1016

    accuracy                         0.9914      3975
   macro avg     0.9875    0.9901    0.9888      3975
weighted avg     0.9915    0.9914    0.9915      3975

Confusion Matrix:


,Pred B,Pred I
Actual B,2938,21
Actual I,13,1003


## 5. Model Comparison: Progression from Rules to Deep Learning

Here is how the performance evolved across the three stages of the project:


In [8]:
comparison_data = [
    {
        "Model": "1. Rule-Based Scansion Algorithm",
        "Approach": "Fixed heuristic (Laghu pair right-to-left, Guru solo)",
        "Token Accuracy": "N/A (Rules)",
        "Exact Grouping Match": "57.30%"
    },
    {
        "Model": "2. Baseline Logistic Regression",
        "Approach": "DictVectorizer (syllables, weights, offset windows)",
        "Token Accuracy": "96.56%",
        "Exact Grouping Match": "90.39%"
    },
    {
        "Model": "3. Deep BiLSTM Sequence Labeler (Ours)",
        "Approach": "Dense embeddings + 2-layer Bidirectional LSTM",
        "Token Accuracy": f"{token_acc * 100:.2f}%",
        "Exact Grouping Match": f"{exact_match_rate * 100:.2f}%"
    }
]

df_comp = pd.DataFrame(comparison_data)
display(df_comp)


,Model,Approach,Token Accuracy,Exact Grouping Match
0,1. Rule-Based Scansion Algorithm,"Fixed heuristic (Laghu pair right-to-left, Gur...",N/A (Rules),57.30%
1,2. Baseline Logistic Regression,"DictVectorizer (syllables, weights, offset win...",96.56%,90.39%
2,3. Deep BiLSTM Sequence Labeler (Ours),Dense embeddings + 2-layer Bidirectional LSTM,99.14%,97.58%


## 6. Inspection of Residual Errors

With ~97% exact match, only ~3% of test words mismatch. Let's inspect some of these remaining errors:


In [9]:
errors = [r for r in test_seq_results if not r["exact_match"]]
print(f"Total test sequence errors: {len(errors)} / {len(test_seq_results)} ({len(errors)/len(test_seq_results)*100:.2f}%)\n")

for r in errors[:10]:
    print(f"Word: {r['word']:12s} | Syllables: {r['syllables']}")
    print(f"  GOLD : {' | '.join(r['gold_groups']):<25} (labels={r['gold_labels']})")
    print(f"  PRED : {' | '.join(r['pred_groups']):<25} (labels={r['pred_labels']})\n")


Total test sequence errors: 32 / 1321 (2.42%)

Word: साहचर्य      | Syllables: ['सा', 'ह', 'च', 'र्य']
  GOLD : साह | चर्य                (labels=['B', 'I', 'B', 'I'])
  PRED : सा | ह | चर्य             (labels=['B', 'B', 'B', 'I'])

Word: बमुक़ाबला     | Syllables: ['ब', 'मु', 'क़ा', 'ब', 'ला']
  GOLD : ब | मु | क़ाब | ला         (labels=['B', 'B', 'B', 'I', 'B'])
  PRED : ब | मु | क़ा | ब | ला      (labels=['B', 'B', 'B', 'B', 'B'])

Word: छिपना        | Syllables: ['छि', 'प', 'ना']
  GOLD : छि | प | ना               (labels=['B', 'B', 'B'])
  PRED : छिप | ना                  (labels=['B', 'I', 'B'])

Word: कुचलना       | Syllables: ['कु', 'च', 'ल', 'ना']
  GOLD : कुच | ल | ना              (labels=['B', 'I', 'B', 'B'])
  PRED : कु | चल | ना              (labels=['B', 'B', 'I', 'B'])

Word: देहात्मवाद   | Syllables: ['दे', 'हा', 'त्म', 'वा', 'द']
  GOLD : दे | हा | त्म | वाद       (labels=['B', 'B', 'B', 'B', 'I'])
  PRED : दे | हात्म | वाद          (labels=['B', 'B', 'I', 'B', 'I'])

Wor

## 7. Interactive Prediction on Any Hindi Word

You can now test the trained Deep BiLSTM model directly on any arbitrary Hindi word!
The function combines classical prosodic syllabification with neural sequence inference.


In [10]:
from hindi_scansion_algorithm import syllabify, HA, NASAL_LATERAL_ONSETS

def assign_weights_and_heaviness(syllables):
    n = len(syllables)
    weights = []
    heaviness = []
    for s in syllables:
        if not s['has_own_vowel']:
            weights.append(1)
            heaviness.append('light')
        elif s['long'] or s['nasal']:
            weights.append(2)
            heaviness.append('intrinsic')
        else:
            weights.append(1)
            heaviness.append('light')

    for idx in range(1, n):
        s = syllables[idx]
        is_mnl_ha_exception = (s['final_cons'] == HA and s['onset_first'] in NASAL_LATERAL_ONSETS)
        if s['lead_conjunct'] >= 1 and s['has_own_vowel'] and not is_mnl_ha_exception:
            weights[idx - 1] = 2
            heaviness[idx - 1] = 'positional'

    return weights, heaviness


def predict_word_bilstm(word):
    word = word.strip()
    syl = syllabify(word)
    if not syl:
        print(f"Could not extract syllables from '{word}'")
        return None

    weights, heaviness = assign_weights_and_heaviness(syl)
    L = len(syl)

    syl_ids = [syl_vocab.get(s['text'], syl_vocab[UNK_TOKEN]) for s in syl]
    w_ids = [weight_map.get(w, 0) for w in weights]
    h_ids = [heavy_map.get(h, 0) for h in heaviness]
    pos_norm = [i / max(1, L - 1) for i in range(L)]

    t_syl = torch.tensor([syl_ids], dtype=torch.long, device=device)
    t_w = torch.tensor([w_ids], dtype=torch.long, device=device)
    t_h = torch.tensor([h_ids], dtype=torch.long, device=device)
    t_p = torch.tensor([pos_norm], dtype=torch.float, device=device).unsqueeze(-1)

    model.eval()
    with torch.no_grad():
        logits = model(t_syl, t_w, t_h, t_p)
        probs = torch.softmax(logits, dim=-1)[0]
        preds = logits[0].argmax(dim=-1).cpu().numpy()

    p_tags = [rev_label_map[idx] for idx in preds]

    # Reconstruct groups
    syllable_groups = []
    weight_groups = []
    curr_syl = []
    curr_w = []

    for s, w, lbl in zip(syl, weights, p_tags):
        if lbl == 'B' and curr_syl:
            syllable_groups.append(curr_syl)
            weight_groups.append(curr_w)
            curr_syl = []
            curr_w = []
        curr_syl.append(s['text'])
        curr_w.append(w)

    if curr_syl:
        syllable_groups.append(curr_syl)
        weight_groups.append(curr_w)

    group_str = " | ".join("".join(g) for g in syllable_groups)
    scansion_str = " | ".join(str(w) for w in weight_groups)

    print("=" * 65)
    print(f"WORD: {word}")
    print("=" * 65)
    detail_rows = []
    for i in range(L):
        detail_rows.append({
            "Pos": i,
            "Syllable": syl[i]['text'],
            "Weight": f"{'Guru' if weights[i] == 2 else 'Laghu'} ({weights[i]})",
            "Heaviness": heaviness[i],
            "BIO Tag": p_tags[i],
            "P(B)": f"{probs[i, 0].item():.4f}",
            "P(I)": f"{probs[i, 1].item():.4f}"
        })
    display(pd.DataFrame(detail_rows))
    print(f"\n  ➤ Formed Word Groups : {group_str}")
    print(f"  ➤ 2-Mātrā Scansion    : {scansion_str}")
    print(f"  ➤ Flat Weight Sequence: {weights}")
    print("=" * 65 + "\n")


# Test on representative words
demo_words = ["कंकरीला", "कक्षा", "बिजली", "निग्रह", "दिवाली", "बात", "नरदेव", "शहसवार"]
for w in demo_words:
    predict_word_bilstm(w)


WORD: कंकरीला


,Pos,Syllable,Weight,Heaviness,BIO Tag,P(B),P(I)
0,0,कं,Guru (2),intrinsic,B,1.0000,0.0000
1,1,क,Laghu (1),light,I,0.0169,0.9831
2,2,री,Guru (2),intrinsic,B,1.0000,0.0000
3,3,ला,Guru (2),intrinsic,B,1.0000,0.0000



  ➤ Formed Word Groups : कंक | री | ला
  ➤ 2-Mātrā Scansion    : [2, 1] | [2] | [2]
  ➤ Flat Weight Sequence: [2, 1, 2, 2]

WORD: कक्षा


,Pos,Syllable,Weight,Heaviness,BIO Tag,P(B),P(I)
0,0,क,Guru (2),positional,B,1.0000,0.0000
1,1,क्षा,Guru (2),intrinsic,B,0.9998,0.0002



  ➤ Formed Word Groups : क | क्षा
  ➤ 2-Mātrā Scansion    : [2] | [2]
  ➤ Flat Weight Sequence: [2, 2]

WORD: बिजली


...


  ➤ Formed Word Groups : बात
  ➤ 2-Mātrā Scansion    : [2, 1]
  ➤ Flat Weight Sequence: [2, 1]

WORD: नरदेव


,Pos,Syllable,Weight,Heaviness,BIO Tag,P(B),P(I)
0,0,न,Laghu (1),light,B,1.0000,0.0000
1,1,र,Laghu (1),light,I,0.0116,0.9884
2,2,दे,Guru (2),intrinsic,B,1.0000,0.0000
3,3,व,Laghu (1),light,I,0.0001,0.9999



  ➤ Formed Word Groups : नर | देव
  ➤ 2-Mātrā Scansion    : [1, 1] | [2, 1]
  ➤ Flat Weight Sequence: [1, 1, 2, 1]

WORD: शहसवार


,Pos,Syllable,Weight,Heaviness,BIO Tag,P(B),P(I)
0,0,श,Laghu (1),light,B,0.9987,0.0013
1,1,ह,Laghu (1),light,B,0.9957,0.0043
2,2,स,Laghu (1),light,I,0.0144,0.9856
3,3,वा,Guru (2),intrinsic,B,1.0000,0.0000
4,4,र,Laghu (1),light,I,0.0001,0.9999



  ➤ Formed Word Groups : श | हस | वार
  ➤ 2-Mātrā Scansion    : [1] | [1, 1] | [2, 1]
  ➤ Flat Weight Sequence: [1, 1, 1, 2, 1]

